<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/(Before-optimisation)_Shor's_Algorithm_for_Factoring_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#Shor's Algorithm for Factoring 15 (with 3 qubits)

#!pip install qiskit # Uncomment installations upon first time running
#!pip install qiskit-aer
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
import numpy as np
import time

# Measures how long the  program takes to run
start_time = time.time()

def shor_factor_15(a=7):
    """
    Real implementation of Shor's algorithm for factoring N=15
    using 3 qubits in the phase estimation register (3-qubit QFT)

    Args:
        a: random base coprime with 15 (options: 2, 4, 7, 8, 11, 13, 14)
    """
    N = 15
    n_count = 3  # 3 qubits for phase estimation

    # Create circuit
    qc = QuantumCircuit(n_count + 4, n_count)  # 3 counting qubits + 4 work qubits

    # Initialize work register to |1>
    qc.x(n_count)  # qubit n_count = |1>

    # Apply Hadamard to counting qubits
    for i in range(n_count):
        qc.h(i)

    # Controlled modular exponentiation for a^x mod 15
    # For a=7 and N=15, the period r=4
    # Implementation of controlled U^(2^k) gates

    # U = multiplication by 7 mod 15
    # U|1> = |7>, U^2|1> = |4>, U^4|1> = |1>

    # Apply controlled gates (simplified version for demonstration)
    # Control qubit 0 (2^0)
    qc.cx(0, n_count)      # controlled swap for demonstration
    qc.cx(0, n_count+1)

    # Control qubit 1 (2^1)
    qc.cx(1, n_count)
    qc.cx(1, n_count+2)

    # Control qubit 2 (2^2)
    qc.cx(2, n_count)
    qc.cx(2, n_count+3)

    # Apply inverse QFT
    qft_circ = QFT(n_count, inverse=True, do_swaps=True)
    qc.append(qft_circ, range(n_count))

    # Measure counting qubits
    qc.measure(range(n_count), range(n_count))

    print("Shor's algorithm circuit for N=15:")
    print(qc)

    # Execute
    backend = Aer.get_backend('qasm_simulator')
    qc_transpiled = transpile(qc, backend)
    job = backend.run(qc_transpiled, shots=2048)
    result = job.result()
    counts = result.get_counts()

    print("\nMeasurement results:")
    print(counts)

    # Process results
    measured_phases = []
    for output, count in counts.items():
        # Convert binary to decimal
        phase = int(output, 2) / (2**n_count)
        measured_phases.append((phase, count))

    # Find period using continued fractions
    possible_r = []
    for phase, count in measured_phases:
        if phase == 0:
            continue
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator
        if r not in [x[0] for x in possible_r] and r > 0:
            possible_r.append((r, count))

    # Sort by frequency
    possible_r.sort(key=lambda x: x[1], reverse=True)

    # Try each candidate r
    for r, freq in possible_r:
        print(f"\nTesting r = {r} (frequency: {freq})")
        if r % 2 == 0:
            # Check if we found a factor
            candidate1 = gcd(a**(r//2) - 1, N)
            candidate2 = gcd(a**(r//2) + 1, N)
            print(f"  Candidates: {candidate1}, {candidate2}")
            if candidate1 not in [1, N] and candidate1 > 1:
                print(f"\n✓ Factor found: {candidate1}")
                return candidate1
            if candidate2 not in [1, N] and candidate2 > 1:
                print(f"\n✓ Factor found: {candidate2}")
                return candidate2

    print("\n✗ No factor found. Try a different value for 'a'.")
    return None

# Execute
print("=== Shor's Algorithm for Factoring 15 ===\n")
factor = shor_factor_15(a=7)
print("Program took %s seconds to run" % (time.time() - start_time))

=== Shor's Algorithm for Factoring 15 ===

Shor's algorithm circuit for N=15:
     ┌───┐                              ┌───────┐┌─┐      
q_0: ┤ H ├──■────■──────────────────────┤0      ├┤M├──────
     ├───┤  │    │                      │       │└╥┘┌─┐   
q_1: ┤ H ├──┼────┼────■────■────────────┤1 IQFT ├─╫─┤M├───
     ├───┤  │    │    │    │            │       │ ║ └╥┘┌─┐
q_2: ┤ H ├──┼────┼────┼────┼────■────■──┤2      ├─╫──╫─┤M├
     ├───┤┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │  └───────┘ ║  ║ └╥┘
q_3: ┤ X ├┤ X ├──┼──┤ X ├──┼──┤ X ├──┼────────────╫──╫──╫─
     └───┘└───┘┌─┴─┐└───┘  │  └───┘  │            ║  ║  ║ 
q_4: ──────────┤ X ├───────┼─────────┼────────────╫──╫──╫─
               └───┘     ┌─┴─┐       │            ║  ║  ║ 
q_5: ────────────────────┤ X ├───────┼────────────╫──╫──╫─
                         └───┘     ┌─┴─┐          ║  ║  ║ 
q_6: ──────────────────────────────┤ X ├──────────╫──╫──╫─
                                   └───┘          ║  ║  ║ 
c: 3/════════════════════════════════

/tmp/ipykernel_48239/2822975954.py:60: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qft_circ = QFT(n_count, inverse=True, do_swaps=True)
